# Human Activity Recognition — Inference / Test Notebook

Takes one Phyphox "Acceleration with g" recording (csv or xls) and prints the
segmented output table required by the project:

| Activity | Time (seconds) | Remark |
|---|---|---|

Uses the models trained and saved by `01_research_notebook.ipynb` (see
`artifacts/`) — run that notebook first (or after adding new training recordings)
to (re)generate them.


In [1]:
import warnings
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

import har_common as hc

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.max_rows", None)

artifacts = hc.load_artifacts()


## Input

Change `INPUT_FILE_PATH` to the recording you want to test. It defaults to the
sample recording in `live_test_file/` for a live demo.


In [2]:
INPUT_FILE_PATH = hc.DEFAULT_TEST_FILE
# INPUT_FILE_PATH = "path/to/your/recording.xls"

raw = hc.load_recording(INPUT_FILE_PATH)
duration = raw["time"].iloc[-1] - raw["time"].iloc[0]
print(f"Loaded {INPUT_FILE_PATH}: {duration:.1f}s, {len(raw)} samples")


Loaded /Users/dan/PycharmProjects/smart_agents_course_project_part_2 2/live_test_file/Raw Data.csv: 49.6s, 23747 samples


## Preprocess + per-window classification

Same resampling/windowing/feature pipeline as training
(`har_common.featurize_recording`), then Random Forest activity prediction,
overridden to "Unknown" when a window is too far (in feature space) from every
trained activity's cluster centroid.


In [3]:
features_df = hc.featurize_recording(raw)
window_labels = hc.classify_windows(features_df, artifacts)
pd.Series(window_labels).value_counts()


still          21
stairs_down     8
walking         6
unknown         2
Name: count, dtype: int64

## Smoothing + segmenting

A small rolling majority-vote filter removes single-window flicker, then
consecutive windows sharing a label are merged into contiguous activity segments —
this is the "identify sequences of activities" step.

Windows overlap by 50% (`OVERLAP=0.5`), so each window's *own* end time already
reaches 1.28s into the next window. Segment boundaries are built from
`slice_end` (each window's non-overlapping slice, ending where the next window
starts) rather than `window_end`, so consecutive segments' durations don't
double-count that overlap -- otherwise every transition would silently add
~1.28s of extra duration and the segment durations wouldn't sum back to the
recording's real length.


In [4]:
smoothed_labels = hc.majority_smooth(window_labels, k=5)
segments = hc.merge_segments(smoothed_labels, features_df["window_start"].tolist(), features_df["slice_end"].tolist())
segments


[{'activity': 'still',
  'start': 0.018361003,
  'end': 30.738361002999998,
  'duration': 30.72},
 {'activity': 'stairs_down',
  'start': 30.738361002999998,
  'end': 32.018361003,
  'duration': 1.2800000000000047},
 {'activity': 'walking',
  'start': 32.018361003,
  'end': 39.698361003,
  'duration': 7.68},
 {'activity': 'stairs_down',
  'start': 39.698361003,
  'end': 48.638361003,
  'duration': 8.939999999999998}]

## Trend detection + final output table

For movement segments (walking / running / stairs up / stairs down) we test
whether the windowed speed-proxy signal has a statistically significant slope
(least-squares regression + significance test) to label Accelerate / Decelerate /
Stable speed. Still and Unknown segments get a blank remark, matching the
project's example table.


In [5]:
output_table = hc.segments_to_table(segments, features_df)
output_table


,Activity,Time (seconds),Remark
0,Still,31,
1,Stairs down,1,Stable speed
2,Walking,8,Stable speed
3,Stairs down,9,Stable speed


In [6]:
print(output_table.to_string(index=False))
print()
recording_duration = raw["time"].iloc[-1] - raw["time"].iloc[0]
print(f"recording length: {recording_duration:.1f}s | sum of reported durations: {output_table['Time (seconds)'].sum()}s")


   Activity  Time (seconds)       Remark
      Still              31             
Stairs down               1 Stable speed
    Walking               8 Stable speed
Stairs down               9 Stable speed

recording length: 49.6s | sum of reported durations: 49s


Segment durations are rounded cumulatively so they sum back to the recording's
covered length (see `har_common.segments_to_table`) rather than drifting from
independent per-segment rounding. The one remaining, expected gap is any
leftover tail shorter than one full window (2.56s) at the very end of the
recording, which can't form a complete window and is simply not covered by any
window — typically under 1 second.
